In [2]:
import torch
import os
import copy
import sys
sys.path.append(os.path.abspath("../data"))
from models.gcn_model import GCN

d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [4]:
def fedavg(models):
    """
    models: list of PyTorch models (LDP-protected)
    """
    global_model = copy.deepcopy(models[0])
    global_state = global_model.state_dict()

    for key in global_state.keys():
        global_state[key] = torch.stack(
            [model.state_dict()[key].float() for model in models],
            dim=0
        ).mean(dim=0)

    global_model.load_state_dict(global_state)
    return global_model

In [5]:
if __name__ == "__main__":

    # Load one graph ONLY to get input_dim (NO DATA USE)
    graph = torch.load("../data/graph/graph_A.pt", weights_only=False)
    input_dim = graph.num_node_features

    # Initialize empty models
    model_A = GCN(input_dim, 32, 2).to(device)
    model_B = GCN(input_dim, 32, 2).to(device)
    model_C = GCN(input_dim, 32, 2).to(device)

    # Load LDP-protected models
    model_A.load_state_dict(torch.load("../data/models/model_A_ldp.pth"))
    model_B.load_state_dict(torch.load("../data/models/model_B_ldp.pth"))
    model_C.load_state_dict(torch.load("../data/models/model_C_ldp.pth"))

    print("LDP-protected local models loaded")

    # FedAvg aggregation
    global_model = fedavg([model_A, model_B, model_C])

    # Save global model
    os.makedirs("../data/models", exist_ok=True)
    torch.save(global_model.state_dict(), "../data/models/global_model.pth")

    print("Global model created using FedAvg")

LDP-protected local models loaded
Global model created using FedAvg
